# Build with CoCo: SaaS Seat Sweeper

Build an internal Finance/IT app, deploy it, add a feature with CoCo, and deploy again. Work individually in your assigned training account. Flagging an assignment requests review: it does not cancel a vendor seat or realize savings.

**How to use this notebook:** read the short instruction, run the next cell or copy its prompt into CoCo chat, then check the result. Run cells one at a time. Setup asks for fresh confirmation at each stage. Markdown prompts do not execute skills.

**Three hours:** introduction 0–5; data and local app 5–30; configure 30–40; deploy 40–65; access 65–80; break 80–90; feature 90–115; redeploy 115–135; operations 135–150; agent demo 150–160; reflection 160–170; cleanup 170–180. These are proposed budgets, not a timed rehearsal.

The [HTML guide](lab-guide.html) is a detailed reference for the older prepared-data track. Follow this notebook's setup order, not both tracks.

## Before the session

Clone the workshop and open its root in CoCo Desktop:

```bash
git clone https://github.com/sfc-gh-jholt/saas-seat-sweeper.git
coco saas-seat-sweeper
```

Open `START_HERE.ipynb` and select a local Python kernel with `snowflake-connector-python` and `ipykernel` installed. The facilitator must verify that kernel, interactive input, your named connection, Node/npm, the delivery CLI, and the `snowflake-apps` skill before the timer starts. Never paste credentials into notebook cells or chat.

Your connection must point to your isolated training account, use its approved authentication, and omit default database/schema/warehouse names that do not exist yet. The notebook connection is separate from CoCo's selected connection and the app's runtime identity.

The facilitator completes platform eligibility and network preparation, **but leaves the four workshop object names unused**. If custom build networking needs the workshop schema, provision it through the approved runbook after setup, before deployment. A restricted test identity can be provisioned beforehand; its workshop role is assigned after step 3. See [preflight](admin-preflight.md) for these checks, not for pre-running the data scripts.

## 1. Connect to your training account

Fill in these three values from your facilitator. Use `ORG-ACCOUNT_NAME`, not the account locator, for `expected_account`. Run the cell and check the returned account, username and `ACCOUNTADMIN` role. This training role is not a production recommendation.

If the folder or import check fails, fix the working directory or selected kernel first. Do not switch to a default connection.

In [ ]:
from pathlib import Path

assert (Path.cwd() / 'scripts/notebook_lab.py').is_file(), 'Set the kernel working directory to the cloned repository root.'
from scripts.notebook_lab import connect_lab

connection_name = 'REPLACE_WITH_NAMED_CONNECTION'
expected_account = 'REPLACE_WITH_ORG-ACCOUNT_NAME'
expected_user = 'REPLACE_WITH_USERNAME'

lab = connect_lab(connection_name, expected_account, expected_user)
lab.check_identity()

## 2. Build the dataset

Open [01_setup.sql](sql/01_setup.sql), then run the next cell. It creates the database, three schemas, an X-Small warehouse and two workshop roles. It checks all four fixed object names before starting and refuses collisions.

Type the exact confirmation shown by the cell to approve that script in your account. If input is unavailable, stop and ask the facilitator to use the reviewed whole-script runbook. **On any error or interruption, stop; DDL can partially commit. Do not reconnect to bypass the guard, reset objects or reseed.**

In [ ]:
lab.bootstrap('01_setup.sql')

Open [02_seed_data.sql](sql/02_seed_data.sql). It creates 2,000 employees, 15 assets, 10 synthetic contracts and randomized assignments. An assignment is one employee/asset pair. Contract text and prices are examples, not current vendor terms.

Run once. Expected result: `Completed 02_seed_data.sql`.

In [ ]:
lab.bootstrap('02_seed_data.sql')

## 3. Add the app-facing views

Open [03_rbac_demo.sql](sql/03_rbac_demo.sql), then run it. This creates six views, a department policy and role grants. The restricted role sees Engineering through department-filtered views; shared contract and inventory views are not department-filtered. It receives no RAW table access.

Expected result: `Completed 03_rbac_demo.sql`. The facilitator can now assign the restricted role to the separate test identity using the approved access process.

In [ ]:
lab.bootstrap('03_rbac_demo.sql')

## 4. Check the data

Run the next two cells. Expect 2,000 employees, 15 assets and 10 contracts; assignments vary. Integrity checks must all return zero. If an assertion fails, stop and inspect rather than rebuilding.

The SQL is inside Python strings because this notebook runs locally; it does not need Snowflake-only SQL magic. Details: [dataset guide](dataset-guide.md).

In [ ]:
counts = lab.query("""
SELECT 'EMPLOYEES' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM SAAS_SEAT_SWEEPER.RAW.EMPLOYEES
UNION ALL SELECT 'ASSETS', COUNT(*) FROM SAAS_SEAT_SWEEPER.RAW.ASSETS
UNION ALL SELECT 'CONTRACTS', COUNT(*) FROM SAAS_SEAT_SWEEPER.RAW.CONTRACTS
UNION ALL SELECT 'EMPLOYEE_ASSETS', COUNT(*) FROM SAAS_SEAT_SWEEPER.RAW.EMPLOYEE_ASSETS
""")
actual = {row['TABLE_NAME']: row['ROW_COUNT'] for row in counts}
assert all(actual[name] == count for name, count in {'EMPLOYEES': 2000, 'ASSETS': 15, 'CONTRACTS': 10}.items()), actual
counts

In [ ]:
checks = lab.query("""
SELECT 'duplicate_employees' AS CHECK_NAME, COUNT(*) AS VIOLATIONS
FROM (SELECT emp_id FROM SAAS_SEAT_SWEEPER.RAW.EMPLOYEES GROUP BY emp_id HAVING COUNT(*) > 1)
UNION ALL SELECT 'duplicate_assets', COUNT(*)
FROM (SELECT asset_id FROM SAAS_SEAT_SWEEPER.RAW.ASSETS GROUP BY asset_id HAVING COUNT(*) > 1)
UNION ALL SELECT 'duplicate_contracts', COUNT(*)
FROM (SELECT contract_id FROM SAAS_SEAT_SWEEPER.RAW.CONTRACTS GROUP BY contract_id HAVING COUNT(*) > 1)
UNION ALL SELECT 'duplicate_assignments', COUNT(*)
FROM (SELECT emp_id, asset_id FROM SAAS_SEAT_SWEEPER.RAW.EMPLOYEE_ASSETS GROUP BY emp_id, asset_id HAVING COUNT(*) > 1)
UNION ALL SELECT 'orphan_employees', COUNT(*) FROM SAAS_SEAT_SWEEPER.RAW.EMPLOYEE_ASSETS ea
LEFT JOIN SAAS_SEAT_SWEEPER.RAW.EMPLOYEES e ON e.emp_id = ea.emp_id WHERE e.emp_id IS NULL
UNION ALL SELECT 'orphan_assets', COUNT(*) FROM SAAS_SEAT_SWEEPER.RAW.EMPLOYEE_ASSETS ea
LEFT JOIN SAAS_SEAT_SWEEPER.RAW.ASSETS a ON a.asset_id = ea.asset_id WHERE a.asset_id IS NULL
UNION ALL SELECT 'orphan_contracts', COUNT(*) FROM SAAS_SEAT_SWEEPER.RAW.ASSETS a
LEFT JOIN SAAS_SEAT_SWEEPER.RAW.CONTRACTS c ON c.contract_id = a.contract_id
WHERE a.contract_id IS NOT NULL AND c.contract_id IS NULL
UNION ALL SELECT 'empty_contract_text', COUNT(*) FROM SAAS_SEAT_SWEEPER.RAW.CONTRACTS
WHERE contract_terms_text IS NULL OR TRIM(contract_terms_text) = ''
""")
assert all(row['VIOLATIONS'] == 0 for row in checks), checks
checks

### Capture your opportunity baseline

The view selects `Inactive` assignments with at least 30 inactive days. The app additionally selects positive monthly seat cost. Sum those assignment costs, then multiply by 12 for annualized opportunity. Status and activity date were sampled independently.

Run this cell before making changes. Keep the result in your private scorecard; your totals need not match another participant's.

In [ ]:
baseline = lab.query("""
SELECT COUNT(*) AS ELIGIBLE_ASSIGNMENTS,
       COALESCE(SUM(monthly_cost_per_seat), 0) AS MONTHLY_OPPORTUNITY
FROM SAAS_SEAT_SWEEPER.APP.V_INACTIVE_USERS_RLS
WHERE monthly_cost_per_seat > 0
""")[0]
assert baseline['ELIGIBLE_ASSIGNMENTS'] > 0, 'No eligible test assignment: ask the facilitator.'
baseline['ANNUALIZED_OPPORTUNITY'] = baseline['MONTHLY_OPPORTUNITY'] * 12
baseline

## 5. Run the starter locally

In a separate terminal, enter `starter-app/`, set `PARTICIPANT_CONNECTION` to the same approved profile, and run:

```bash
: "${PARTICIPANT_CONNECTION:?Set your approved named connection first}"
snow connection test --connection "$PARTICIPANT_CONNECTION"
# Check the account and user above before continuing.
npm ci
SNOWFLAKE_CONNECTION_NAME="$PARTICIPANT_CONNECTION" SNOWFLAKE_DEFAULT_CONNECTION_NAME="$PARTICIPANT_CONNECTION" SNOWFLAKE_ROLE=ACCOUNTADMIN SNOWFLAKE_WAREHOUSE=SAAS_SWEEPER_WH npm run dev
```

Keep that terminal open. Open the printed localhost URL, then Overview, Licenses, Inventory and Ask. Compare Overview's opportunity to your baseline and open one contract. Licenses is read-only; Ask describes the later demo.

The app can use environment credentials ahead of the named profile. Have the facilitator check unintended overrides without printing secrets. A working notebook connection does not prove the app uses the same identity. If totals are zero, inspect errors before accepting them.

**Check:** all four pages load and the baseline matches. For launch troubleshooting only, use [Module 01](lab-instructions/01-run-starter.md); do not rerun its prepared-data steps.

## 6. Use a skill: deploy version one

Copy the prompt below into CoCo chat. `snowflake-apps` is the skill; `snow app deploy` is a CLI command it may use. Configuration has 10 minutes (30–40), followed by deployment (40–65). Review the target and approve only the changes you intend.

<!-- prompt: coco-prompts/notebook-deploy.md -->
```text
Use the snowflake-apps skill to configure and deploy starter-app/ to my assigned workshop account. Ask for my named connection and unused app name if they are not already in this conversation. Verify the account explicitly; do not use a default connection or the reference app.

Use SAAS_SEAT_SWEEPER.APP and SAAS_SWEEPER_WH. Preserve snowflake.yml for deployment and build-only app.yml; do not migrate manifests or change account policies. Show the exact target and configuration changes, then wait for my approval before setup or deployment. Follow the skill's required checks, keeping explanations short. After deployment, return the actual URL, version and status; stop on failure rather than retrying automatically.
```

**Check:** open the returned remote URL, not localhost. Visit the four pages and compare opportunity totals again. Record the actual service name and deployed version privately. At minute 65, mark a blocked deployment honestly and use the [facilitator checkpoint](deployment-checkpoint.md); a demonstration URL is not your deployment.

## 7. Check access (65–80)

With the facilitator, follow [the access exercise](lab-instructions/04-access.md) against your exact service. Test entry using the separate non-admin browser identity before and after an approved grant/revoke. If that identity is unavailable, mark the proof blocked.

Separately, use the restricted-role SQL checks in [03_rbac_demo.sql](sql/03_rbac_demo.sql) in a dedicated worksheet: disable secondary roles, select the restricted role, read the department view, and attempt the supplied zero-row update. Expect Engineering only and an authorization error on the update. Restore the worksheet's previous context.

**Check:** distinguish app entry from data access. The app queries with its owner/service identity; allowing a visitor in does not make SQL run as that visitor. Keep this notebook's administrator session unchanged.

## Break (80–90)

Take the full ten minutes.

## 8. Add your feature (90–115)

Ask CoCo for a small plan, approve it, then review the diff and local tests. This is the main AI-assisted development exercise.

<!-- prompt: coco-prompts/notebook-feature.md -->
```text
Add a Flag for review action to the existing starter-app Licenses page. Inspect only the relevant source and the Technical checklist in lab-instructions/06-build-reclamation.md. Propose the smallest change and tests in five bullets, then wait for approval before editing.

Use a parameterized conditional Inactive-to-Flagged update for one employee/asset pair. Validate positive safe-integer IDs and JSON, parse the affected-row summary correctly, distinguish success/no-op/failure, and refresh authoritative rows and totals. Flagging is not cancellation or realized savings. Preserve authorization, manifests and the informational Ask page. After approval, implement and run local tests; do not deploy or send live mutations yet.
```

**Check:** a focused diff and passing local tests, not a replacement application. Stop implementation at minute 110; use the [reviewed feature checkpoint](checkpoints/README.md) if needed. A checkpoint counts as recovery, not code you implemented. Save live write verification for the deployed version so you do not perform the same mutation exercise twice.

## 9. Redeploy and verify (115–135)

<!-- prompt: coco-prompts/notebook-redeploy.md -->
```text
Use the snowflake-apps skill to redeploy the tested starter-app change to the same reviewed participant service. Reuse this conversation's target evidence, recheck the live account and manifest, and stop if either changed. Show the proposed deployment and wait for approval. Keep policies, integrations, manifest layout and Ask unchanged. Return the actual URL/version/status; do not automatically retry or send a reclamation request.
```

Open the returned remote version. With CoCo, choose one eligible assignment and inspect its stored state. Review the **exact endpoint, employee/asset IDs and one request executor** before approving a live test. Localhost also writes to remote Snowflake; it is not a sandbox.

Flag the chosen assignment through the app's HTTP action, then replay the identical request once. Expect one change followed by a no-op. Verify stored `Flagged` status and refreshed opportunity totals. Do not substitute a direct SQL update or click a different assignment. If the response is ambiguous, inspect state before retrying.

**Check:** deployed version, one changed assignment, one no-op, and matching refreshed data. See [verification details](lab-instructions/07-deploy-v2.md) if needed. At minute 135, record any remaining blocker and move on.

## 10. Inspect operations (135–150)

Use the existing deployment conversation to inspect your service's status and recent events read-only. Pick one build or request event and trace what happened. Do not restart resources or change scaling just to create an incident.

**Check:** explain which evidence describes app runtime and which describes SQL warehouse work. They are separate compute/cost boundaries. [Operations reference](lab-instructions/08-operate.md).

## 11. Watch the business agent (150–160)

The facilitator demonstrates one structured question, one contract question and one combined question using a separately prepared Cortex Agent. Compare a number with SQL and inspect a contract citation.

**Check:** distinguish CoCo, which helped build the app, from the business agent. The starter Ask page remains informational. No participant PAT, AI setup or secret mount is required. [Demo reference](lab-instructions/09-ask-demo.md).

## 12. Record what you proved (160–170)

In your private [scorecard](scorecard.md), record the two deployed versions, the feature result, one useful skill, and any blocked checks. Explain why potential savings differ from realized savings. A build success alone is not proof of correct data or access.

## 13. Clean up (170–180)

Follow [the cleanup guide](teardown-guide.md) for your recorded participant service only. Review the exact target and dependents before approving removal. Do not drop the shared data database, roles, warehouse, integrations or any reference service as a shortcut. The facilitator/cost owner decides whether to retain or remove those remaining resources.

Stop your local app terminal with Ctrl+C. Run the cell below to close the notebook connection. Closing a connection does not delete or suspend all billable resources.

Before sharing or committing this notebook, restore the `REPLACE_...` connection placeholders and clear **all outputs and execution counts**. Run `python3 scripts/check_repository.py`; publication checks reject executed notebooks. Keep account IDs, endpoints and evidence outside the repository.

In [ ]:
lab.close()